In [4]:
import ydf #YDF is available on Python 3.8, 3.9, 3.10, 3.11 and 3.12, on Windows x86-64 Linux x86-64, and macOS ARM64.
import pandas as pd
import numpy as np
# run with cleaned dataset and without changing frequency of non IC

In [5]:
crime = pd.read_csv("../Crime_Data_from_2020_to_2024.csv")
crime.columns = crime.columns.str.replace(' ', '_')
crime.columns

Index(['DR_NO', 'Date_Rptd', 'DATE_OCC', 'TIME_OCC', 'AREA', 'AREA_NAME',
       'Rpt_Dist_No', 'Part_1-2', 'Crm_Cd', 'Crm_Cd_Desc', 'Mocodes',
       'Vict_Age', 'Vict_Sex', 'Vict_Descent', 'Premis_Cd', 'Premis_Desc',
       'Weapon_Used_Cd', 'Weapon_Desc', 'Status', 'Status_Desc', 'Crm_Cd_1',
       'Crm_Cd_2', 'Crm_Cd_3', 'Crm_Cd_4', 'LOCATION', 'Cross_Street', 'LAT',
       'LON'],
      dtype='str')

In [ ]:
# Drop redundant/unnecessary columns
crime= crime.drop(columns=["DR_NO", "Date_Rptd", "AREA","Part_1-2","Crm_Cd", 
                   "Mocodes","Status_Desc", "Premis_Cd","Weapon_Used_Cd", 
                   "LOCATION","Cross_Street"])
#crime.columns

Index(['DATE_OCC', 'TIME_OCC', 'AREA_NAME', 'Rpt_Dist_No', 'Crm_Cd_Desc',
       'Vict_Age', 'Vict_Sex', 'Vict_Descent', 'Premis_Desc', 'Weapon_Desc',
       'Status', 'Crm_Cd_1', 'Crm_Cd_2', 'Crm_Cd_3', 'Crm_Cd_4', 'LAT', 'LON'],
      dtype='str')

In [7]:
# Remove Missing Labels + CC
crime = crime.dropna(subset=['Status'])
crime = crime[crime['Status'] != 'CC']

In [8]:
# Set missing values to unknown (X)
crime['Vict_Sex'] = crime['Vict_Sex'].replace(np.nan, "X")
crime['Vict_Sex'] = crime['Vict_Sex'].replace("-", "X")
crime['Vict_Sex'] = crime['Vict_Sex'].replace("H", "X")

crime['Vict_Descent'] = crime['Vict_Descent'].replace(np.nan, "X")
crime['Vict_Descent'] = crime['Vict_Descent'].replace("-", "X")


In [9]:
# Remove negative ages
crime.loc[crime['Vict_Age'] < 0, 'Vict_Age'] = np.nan

In [ ]:
#Force strings
crime["Premis_Desc"] = (crime["Premis_Desc"].fillna("UNKNOWN").apply(lambda x: str(x)))
crime["Weapon_Desc"] = (crime["Weapon_Desc"].fillna("UNKNOWN").apply(lambda x: str(x)))

Weapon_Desc
<class 'str'>    1004887
Name: count, dtype: int64

In [11]:
def split_dataset(ds, test_ratio):
  test_indices = np.random.rand(len(ds)) < test_ratio
  return ds[~test_indices], ds[test_indices]

crime_train, crime_test = split_dataset(crime, 0.10)
#print(len(crime_train), len(crime_test))


In [12]:
model = ydf.RandomForestLearner(label = "Status", num_trees = 20).train(crime_train)

Train model on 904682 examples
Model trained in 0:02:10.288115


In [13]:
model.evaluate(crime_test)

Label \ Pred,IC,AO,AA,JA,JO
IC,77214,6272,5587,244,144
AO,2298,4216,1917,25,34
AA,482,482,1233,50,0
JA,0,0,0,4,0
JO,1,0,0,0,2


In [14]:
model.analyze(crime_test)